# PA5: Scalable, Enterprise-Grade Agentic AI Game Development
## Extension of Dino Runner base architecture with ReAct Agents, Guardrails, Memory Management, and MLflow Tracking.
**Roll Number:** 25280019

## Task 0: Workspace Initialization & Setup

In [0]:
%pip install --upgrade typing_extensions>=4.12.0
%pip install --upgrade databricks-langchain
%pip install --upgrade langchain langchain_community langgraph langchain-experimental pygame
%pip install mlflow presidio-analyzer presidio-anonymizer
dbutils.library.restartPython()

  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.10.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6b0901b1-8610-4ff7-90de-b60b283cea5b
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.

*** WARNING: max output size exceeded, skipping output. ***

610-4ff7-90de-b60b283cea5b/lib/python3.11/site-packages (from mlflow) (6.0.2)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.1/32.1 MB 207.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 k

## Imports, API Setup & LLM Initialization

In [0]:
import os
import sys
import json
import time
import mlflow
import subprocess
from typing import Callable, TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine

# Fix authentication for serverless compute
try:
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    os.environ['DATABRICKS_TOKEN'] = token
    os.environ['DATABRICKS_HOST'] = host
except Exception as e:
    print(f"Warning: Could not extract token: {e}")

# Initialize LLM - Updated to use Llama 4 Maverick (400B params, 128K context)
llm = ChatDatabricks(endpoint="databricks-llama-4-maverick")

# Initialize Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# Custom Pattern Recognizer for Passwords
password_pattern = Pattern(
    name="password_pattern",
    regex=r"(?i)\b(password|passphrase|pwd|passwd|secret)\s*[:=]\s*[^\s]{6,}\b",
    score=0.85
)
password_recognizer = PatternRecognizer(
    supported_entity="PASSWORD",
    patterns=[password_pattern]
)
analyzer.registry.add_recognizer(password_recognizer)

# Custom Pattern Recognizer for API Keys (generic Developer API key pattern)
api_key_pattern = Pattern(
    name="api_key_pattern",
    regex=r"\b((?:api|db|secret|auth|client|access|token|key)_[a-zA-Z0-9_\-]{16,})\b",
    score=0.85
)
api_key_recognizer = PatternRecognizer(
    supported_entity="API_KEY",
    patterns=[api_key_pattern]
)
analyzer.registry.add_recognizer(api_key_recognizer)

/databricks/python/lib/python3.11/site-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 107.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in an interactive Python session, you may need to exit and restart
Python to load all the package's dependencies. You can exit with Ctrl-D (or
Ctrl-Z and Enter on Windows).


## Game State Definition

In [0]:
class GameState(TypedDict):
    director_messages: List[BaseMessage]
    architect_messages: List[BaseMessage]
    engineer_code: str
    qa_feedback: List[BaseMessage]
    current_actor: str
    iteration: int
    iteration_score: List[int]
    latencies: dict  # To track latency for MLflow
    satisfied: bool

## Task 1: ReAct Agent Architecture & Tool Integration
We develop a custom `CodeInterpreterTool` that handles headless Pygame execution using the dummy video driver, captures compilation and runtime outputs, and handles timeout errors safely.

In [0]:
@tool
def code_interpreter_tool(code: str) -> str:
    """
    Executes the generated Python code block headlessly to test for syntax, import, or runtime errors.
    Returns the execution trace (stdout/stderr) and exit status.
    """
    import os
    import sys
    import subprocess
    
    test_filename = "temp_test_code.py"
    with open(test_filename, "w", encoding="utf-8") as f:
        f.write(code)
        
    # Configure headless execution for pygame (so it doesn't fail on displayless Databricks nodes)
    env = os.environ.copy()
    env["SDL_VIDEODRIVER"] = "dummy"
    env["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"
    
    try:
        # Run code for at most 5 seconds to test initialization and game loop startup
        result = subprocess.run(
            [sys.executable, test_filename],
            capture_output=True,
            text=True,
            env=env,
            timeout=5
        )
        output = f"Exit Code: {result.returncode}\nStdout: {result.stdout}\nStderr: {result.stderr}"
        if result.returncode == 0:
            return f"SUCCESS: Code executed cleanly with exit code 0.\n{output}"
        else:
            return f"FAILURE: Code crashed during execution.\n{output}"
            
    except subprocess.TimeoutExpired as e:
        # Timeout is expected behavior because Pygame has an infinite loop while running
        stdout = e.stdout.decode() if e.stdout else ""
        stderr = e.stderr.decode() if e.stderr else ""
        return (
            "SUCCESS: Code successfully initialized and entered Pygame main loop (terminated via timeout as expected).\n"
            f"Stdout: {stdout}\nStderr: {stderr}"
        )
    except Exception as e:
        return f"FAILURE: Execution process error: {str(e)}"
    finally:
        if os.path.exists(test_filename):
            try:
                os.remove(test_filename)
            except Exception:
                pass

## Task 2: Implement Nodes & Guardrails

In [0]:
def director_node(state: GameState):
    start_time = time.time()
    director_msg = input("Awaiting Director Prompt: ")
    
    # Analyze and redact PII entities (standard + custom password & api_key)
    results = analyzer.analyze(
        text=director_msg, 
        entities=["EMAIL_ADDRESS", "IP_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "PASSWORD", "API_KEY"], 
        language='en'
    )
    anonymized = anonymizer.anonymize(text=director_msg, analyzer_results=results)
    safe_text = anonymized.text
    
    if safe_text != director_msg:
        report = {
            "original": director_msg, 
            "redacted": safe_text, 
            "entities": [res.entity_type for res in results]
        }
        with open("pii_redaction_report.json", "w") as f:
            json.dump(report, f, indent=4)
        print("\n[Guardrail] PII detected and redacted. Report saved to 'pii_redaction_report.json'.")
        
    latencies = state.get("latencies", {})
    latencies["director"] = time.time() - start_time
    
    # Initialize history list or append to existing list
    history = state.get("director_messages", [])
    return {
        "director_messages": history + [HumanMessage(content=safe_text)], 
        "current_actor": "director", 
        "latencies": latencies
    }

In [0]:
def architect_node(state: GameState):
    start_time = time.time()
    current_iter = state.get("iteration", 0) + 1
    print(f"\n========== ITERATION {current_iter} ==========")
    
    # Retrieve director input (and all historical feedback)
    director_content = "\n".join([m.content for m in state.get("director_messages", [])])
    
    system_msg = SystemMessage(content=(
        "You are an expert Software Architect. Design the Dino Runner game system architecture. "
        "Create a detailed plan detailing object-oriented layout and state definitions. "
        "CRITICAL REQUIREMENTS:\n"
        "1. Strictly apply OOP principles (separate classes for Dino, Obstacles, and Score).\n"
        
        "2. Dino positioning and mechanics:\n"
        "   - MUST be positioned at ground level: self.y = SCREEN_HEIGHT - self.height\n"
        "   - Standing height: 50px, Ducking height: 25px\n"
        "   - Colors: Dino=Grey RGB(100,100,100), must be visually distinct\n"
        "   - Ducking: pygame.KEYDOWN K_DOWN reduces height to 25, adjusts y to stay grounded\n"
        "   - Standing: pygame.KEYUP K_DOWN restores height to 50\n"
        "   - Jumping: pygame.KEYDOWN K_SPACE or pygame.KEYDOWN K_UP triggers self.dino.jump()\n"
        "   - **CRITICAL Jump Physics Bug Prevention**: In update() method, NEVER reset self.y = SCREEN_HEIGHT - self.height unconditionally every frame. Only set y to ground position when the dino is actually on the ground (after gravity is applied). The correct flow is: (1) Apply gravity/velocity to y, (2) Check if y >= ground position, (3) ONLY THEN clamp y to ground and reset velocity.\n"
        
        "3. Obstacle Spawning System (CRITICAL) & Clustered Obstacles:\n"
        "   - Define a SINGLE custom event: SPAWN_OBSTACLE = pygame.USEREVENT + 1\n"
        "   - Set timer ONCE in __init__: pygame.time.set_timer(SPAWN_OBSTACLE, 2000)  # 2 seconds\n"
        "   - In event loop, when event.type == SPAWN_OBSTACLE and game_active is True:\n"
        "     * Randomly select to spawn either a 'cactus' or a 'pterodactyl' (never both simultaneously).\n"
        "     * If 'pterodactyl' is selected, spawn a single Pterodactyl at x = SCREEN_WIDTH, at a random altitude (e.g. SCREEN_HEIGHT-150, SCREEN_HEIGHT-100, or SCREEN_HEIGHT-70), Red RGB(200,0,0).\n"
        "     * If 'cactus' is selected, spawn a CLUSTER of 1, 2, or 3 Cacti close together. Each cactus in the cluster should be a separate Cactus object, spawned at x positions offset by a small spacing so they appear adjacent but do not overlap (e.g., Cactus 1 at x = SCREEN_WIDTH, Cactus 2 at x = SCREEN_WIDTH + 35, Cactus 3 at x = SCREEN_WIDTH + 70, where each Cactus has a width of 30px). They must all be at y = SCREEN_HEIGHT - height (ground level), Green RGB(0,150,0).\n"
        "   - In game loop update: Remove obstacles when x + width < 0 (off left edge): obstacles = [obs for obs in obstacles if obs.x + obs.width > 0]\n"
        "   - This ensures NEVER spawning both types simultaneously and proper memory cleanup\n"
        
        "4. Day/Night Cycle:\n"
        "   - Day mode: Sky blue RGB(135,206,235)\n"
        "   - Night mode: Midnight blue RGB(25,25,112)\n"
        "   - Toggle every 300 frames or based on score milestones\n"
        
        "5. Score Logic (CRITICAL):\n"
        "   - DO NOT use timers for scoring\n"
        "   - Score increments by 1 ONLY when: obstacle.x + obstacle.width < dino.x AND obstacle.passed == False\n"
        "   - Then set obstacle.passed = True to prevent double-counting\n"
        
        "6. Speed progression: Start at speed=6. As the score increases, speed should increase gradually. Formula: self.speed = 6 + (self.score.current_score / 15.0). To keep the game playable, cap the speed at a maximum of 15 (e.g., self.speed = min(15, 6 + (self.score.current_score / 15.0))).\n"
        
        "7. High-Score Persistence: Save and load the high score from a file using a path relative to the script itself: os.path.join(os.path.dirname(os.path.abspath(__file__)), 'highscore.txt'). Load the score in Score.__init__, strip whitespace, and handle both FileNotFoundError and ValueError gracefully (defaulting to 0). Immediately save the new high score to the file in real-time whenever the current score exceeds the high score. Also, save the high score when the player exits the game (pygame.QUIT event).\n"
        
        "8. Game Over & Reset:\n"
        "   - Display 'GAME OVER' in red at center, with 'Press SPACE' instruction\n"
        "   - SPACE key triggers complete reset: clear obstacles list, reset score/speed, game_active=True\n"
        "   - Design with a SINGLE main event loop at the start of the game loop to handle events (including restarting) for both active and game-over states to prevent event starvation.\n\n"
        
        "Output the architectural requirements with these exact specifications."
    ))
    
    response = llm.invoke([system_msg, HumanMessage(content=f"Design Requirements & Feedback:\n{director_content}")])
    
    latencies = state.get("latencies", {})
    latencies[f"architect_iter_{current_iter}"] = time.time() - start_time
    
    history = state.get("architect_messages", [])
    return {
        "architect_messages": history + [response], 
        "current_actor": "architect", 
        "latencies": latencies
    }

In [0]:
# Simplified Engineer Node - Direct LLM call without ReAct agent
def engineer_node(state: GameState):
    start_time = time.time()
    current_iter = state.get("iteration", 0) + 1
    
    architect_design = state.get("architect_messages", [])[-1].content
    qa_feedback = state.get("qa_feedback", [])
    
    qa_context = ""
    if qa_feedback:
        qa_context = f"\nPrevious QA Feedback/Bugs to Fix:\n{qa_feedback[-1].content}"
    
    system_instructions = (
        "You are a Senior Game Developer. Generate complete, working Python Pygame code for a Dino Runner game.\n\n"
        "CRITICAL REQUIREMENTS:\n"
        "1. Use strict OOP with classes: Dino, Obstacle (base), Cactus(Obstacle), Pterodactyl(Obstacle), Score, Game\n"
        
        "2. Dino class - CRITICAL jump physics fix:\n"
        "   - Color: RGB(100, 100, 100) grey\n"
        "   - Initial position: x=50, y=SCREEN_HEIGHT-50, width=40, height=50\n"
        "   - Jump physics: self.gravity=1, self.velocity=0, jump_velocity=-15\n"
        "   - **CRITICAL update() method implementation** (THIS FIXES THE JUMP BUG):\n"
        "     ```python\n"
        "     def update(self):\n"
        "         # Step 1: Apply gravity and velocity FIRST\n"
        "         self.velocity += self.gravity\n"
        "         self.y += self.velocity\n"
        "         \n"
        "         # Step 2: Calculate ground position based on current height\n"
        "         ground_y = SCREEN_HEIGHT - self.height\n"
        "         \n"
        "         # Step 3: ONLY clamp to ground if we've hit/passed it\n"
        "         if self.y >= ground_y:\n"
        "             self.y = ground_y\n"
        "             self.velocity = 0\n"
        "     ```\n"
        "   - jump() method: if self.y == SCREEN_HEIGHT - self.height: self.velocity = -15\n"
        "   - duck() method (triggered on KEYDOWN of K_DOWN): self.height = 25\n"
        "   - stand() method (triggered on KEYUP of K_DOWN): self.height = 50\n"
        "   - DO NOT modify self.y in duck() or stand() methods\n"
        
        "3. Obstacle Spawning System & Clustered Obstacles - CRITICAL implementation:\n"
        "   - Define custom event: SPAWN_OBSTACLE = pygame.USEREVENT + 1\n"
        "   - In __init__, set timer ONCE: pygame.time.set_timer(SPAWN_OBSTACLE, 2000)\n"
        "   - Obstacle list: self.obstacles = []\n"
        "   - **In event loop, add this EXACT pattern**:\n"
        "     ```python\n"
        "     if event.type == SPAWN_OBSTACLE and self.game_active:\n"
        "         obstacle_type = random.choice(['cactus', 'pterodactyl'])\n"
        "         if obstacle_type == 'cactus':\n"
        "             cluster_size = random.randint(1, 3)\n"
        "             for i in range(cluster_size):\n"
        "                 self.obstacles.append(Cactus(SCREEN_WIDTH + i * 35, SCREEN_HEIGHT))\n"
        "         else:\n"
        "             self.obstacles.append(Pterodactyl(SCREEN_WIDTH, SCREEN_HEIGHT))\n"
        "     ```\n"
        "   - Cactus.__init__: x=start_x, y=screen_height-40, width=30, height=40, color=(0,150,0), self.passed=False\n"
        "   - Pterodactyl.__init__: x=start_x, y=random.choice([screen_height-150, screen_height-100, screen_height-70]), width=50, height=30, color=(200,0,0), self.passed=False\n"
        "   - **In game loop, add cleanup** (CRITICAL to prevent memory leak):\n"
        "     ```python\n"
        "     # Remove off-screen obstacles\n"
        "     self.obstacles = [obs for obs in self.obstacles if obs.x + obs.width > 0]\n"
        "     ```\n"
        
        "4. Score Logic - CRITICAL implementation:\n"
        "   - In game loop, for each obstacle:\n"
        "     ```python\n"
        "     if not obs.passed and obs.x + obs.width < dino.x:\n"
        "         obs.passed = True\n"
        "         self.score += 1\n"
        "     ```\n"
        "   - Display both current score and high score at top-left in WHITE color: font.render(f'Score: {score}  HI: {high_score}', True, (255,255,255))\n"
        
        "5. Speed: self.speed = min(15, 6 + (self.score.current_score / 15.0))\n"
        
        "6. High Score: Load/save high score using a script-relative path: `os.path.join(os.path.dirname(os.path.abspath(__file__)), 'highscore.txt')`. Catch both FileNotFoundError and ValueError during initialization, stripping file contents. Immediately save the high score in real-time when the current score exceeds the high score. Also save the high score to the file inside the `pygame.QUIT` event handling block.\n"
        
        "7. Day/Night: self.frame_count += 1; if self.frame_count % 300 == 0: toggle between RGB(135,206,235) and RGB(25,25,112)\n"
        
        "8. Game Over:\n"
        "   - When collision: self.game_active = False, save high score\n"
        "   - Display 'GAME OVER' in RED RGB(255,0,0) at center\n"
        "   - Wait for SPACE: if event.key == K_SPACE and not self.game_active: reset everything\n"
        "   - Reset MUST: clear self.obstacles = [], self.score = 0, self.speed = 6, self.game_active = True\n\n"
        
        "CODE STRUCTURE REQUIREMENTS:\n"
        "- Screen: 800x400\n"
        "- Use pygame.Rect for all collision detection\n"
        "- Use pygame.draw.rect() with exact RGB colors specified above\n"
        "- Event handling in a SINGLE event loop at the start of the main game loop (do NOT place a second event loop in the game-over block, as this leads to event starvation and prevents space bar restart from working), NOT in update() methods.\n"
        "- **CRITICAL Event Key Check**: Inside the event loop, you must ONLY check `event.key` (such as pygame.K_SPACE, pygame.K_DOWN, pygame.K_UP) nested inside the `event.type == pygame.KEYDOWN` or `event.type == pygame.KEYUP` block. Never check `event.key` at the outer event loop level as this will crash the game with an `AttributeError: 'pygame.event.Event' object has no attribute 'key'` when handling other event types.\n"
        "- **Event Controls**: Bind jumping to K_SPACE and K_UP keys under KEYDOWN when game_active is True. Bind ducking to K_DOWN key under KEYDOWN when game_active is True. Bind standing up to release of K_DOWN (KEYUP) when game_active is True. Bind game restart/reset to K_SPACE under KEYDOWN when game_active is False.\n\n"
        
        "Return ONLY the complete Python code, no explanations. Code must be production-ready."
    )
    
    prompt = f"{system_instructions}\n\nArchitect Design:\n{architect_design}\n{qa_context}\n\nGenerate the complete Python Pygame code:"
    
    try:
        # Direct LLM call - single invocation, no ReAct loop
        response = llm.invoke([
            SystemMessage(content="You are an expert Python game developer. Output ONLY code."),
            HumanMessage(content=prompt)
        ])
        final_output = response.content
        print(f"\n[Engineer Node] Code generated successfully ({len(final_output)} chars)")
    except Exception as e:
        print(f"\n[Engineer Node Error] LLM invocation failed: {type(e).__name__}: {e}")
        print("[Engineer Node] Using fallback: returning previous code or empty placeholder")
        # Use previous iteration's code as fallback
        prev_code = state.get("engineer_code", "")
        if prev_code:
            final_output = f"```python\n{prev_code}\n```"
        else:
            # Simple placeholder if no previous code exists
            final_output = "```python\n# Code generation failed - no code available\nprint('Engineer node failed to generate code')\n```"
    
    # Extract code block
    code = final_output
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0].strip()
    elif "```" in code:
        code = code.split("```")[1].split("```")[0].strip()
        
    latencies = state.get("latencies", {})
    latencies[f"engineer_iter_{current_iter}"] = time.time() - start_time
    
    return {
        "engineer_code": code, 
        "current_actor": "engineer", 
        "iteration": current_iter, 
        "latencies": latencies
    }

In [0]:
# Simplified QA Node - Direct LLM call without ReAct agent
def qa_node(state: GameState):
    start_time = time.time()
    current_iter = state.get("iteration", 0)
    
    architect_design = state.get("architect_messages", [])[-1].content
    engineer_code = state.get("engineer_code", "")
    
    system_instructions = (
        "You are an expert QA Engineer. Evaluate the Pygame code against requirements.\n\n"
        "CRITICAL VALIDATION CHECKLIST:\n"
        "1. OOP design with proper classes (Dino, Obstacles, Score)\n"
        
        "2. POSITIONING (HIGH PRIORITY):\n"
        "   - Verify Dino initial position: y = SCREEN_HEIGHT - height\n"
        "   - Verify Cacti spawn at: y = SCREEN_HEIGHT - height\n"
        "   - Verify Pterodactyls spawn at elevated positions (NOT ground level)\n"
        
        "3. COLOR CODING (HIGH PRIORITY):\n"
        "   - Dino: Grey RGB(100,100,100)\n"
        "   - Cacti: Green RGB(0,150,0)\n"
        "   - Pterodactyls: Red RGB(200,0,0)\n"
        "   - Check pygame.draw.rect uses these exact RGB values\n"
        
        "4. SPAWN LOGIC (HIGH PRIORITY):\n"
        "   - PASS: Single SPAWN_OBSTACLE event defined as pygame.USEREVENT + 1\n"
        "   - PASS: pygame.time.set_timer called ONCE in __init__ with 2000ms\n"
        "   - PASS: Event handler uses random.choice(['cactus', 'pterodactyl']) to pick ONE type\n"
        "   - PASS: If 'cactus' is chosen, it spawns a cluster of 1 to 3 cacti offset by a small spacing (e.g. 35px)\n"
        "   - PASS: Spawning only happens when game_active == True\n"
        "   - PASS: Off-screen obstacle cleanup present (obstacles = [obs for obs in self.obstacles if obs.x + obs.width > 0])\n"
        "   - FAIL: If spawning both types in same event, or no cleanup code, or timer not set, or no cactus clustering\n"
        
        "5. JUMP PHYSICS (HIGH PRIORITY):\n"
        "   - PASS: update() applies velocity BEFORE checking ground: self.velocity += self.gravity; self.y += self.velocity\n"
        "   - PASS: Ground clamping ONLY happens when self.y >= ground_y\n"
        "   - FAIL: If update() unconditionally sets self.y = SCREEN_HEIGHT - self.height every frame\n"
        "   - PASS: duck() and stand() only change self.height, NOT self.y\n"
        
        "6. Day/Night background transitions (RGB color changes every 300 frames)\n"
        
        "7. Speed increase formula: capped at 15, e.g., min(15, 6 + (score / 15.0))\n"
        
        "8. High-score persistence: loaded from and saved to a path relative to the script using os.path.dirname, catching FileNotFoundError and ValueError, updated in real-time and saved on QUIT and collision.\n"
        
        "9. Score incrementing: ONLY when obstacle.x + obstacle.width < dino.x with 'passed' flag\n"
        
        "10. Game Over screen and SPACE reset (clear obstacles list)\n\n"
        
        "Provide a concise report (max 300 words) with:\n"
        "- PASS/FAIL status for positioning, colors, spawn logic, and jump physics\n"
        "- Specific bugs found (quote code lines)\n"
        "- What works correctly\n"
        "- Overall assessment"
    )
    
    prompt = f"{system_instructions}\n\nCode to evaluate (truncated for brevity):\n```python\n{engineer_code[:2000]}\n...\n```\n\nProvide QA report:"
    
    try:
        # Direct LLM call - single invocation
        response = llm.invoke([
            SystemMessage(content="You are an expert QA engineer."),
            HumanMessage(content=prompt)
        ])
        qa_report = response.content
        print(f"\n[QA Node] Report generated successfully ({len(qa_report)} chars)")
    except Exception as e:
        print(f"\n[QA Node Error] LLM invocation failed: {type(e).__name__}: {e}")
        print("[QA Node] Using fallback report")
        qa_report = (
            f"QA Report (Fallback - LLM unavailable):\n\n"
            f"Code evaluation could not be completed due to API error.\n"
            f"Code length: {len(engineer_code)} characters\n"
            f"Manual review recommended.\n\n"
            f"Error: {str(e)[:100]}"
        )
    
    latencies = state.get("latencies", {})
    latencies[f"qa_iter_{current_iter}"] = time.time() - start_time
    
    history = state.get("qa_feedback", [])
    return {
        "qa_feedback": history + [HumanMessage(content=qa_report)], 
        "current_actor": "qa", 
        "latencies": latencies
    }

class ScoreOutput(BaseModel):
    score: int = Field(description="Integrity/completeness score from 1 to 10")

def score_node(state: GameState):
    qa_report = state.get("qa_feedback", [])[-1].content
    structured_llm = llm.with_structured_output(ScoreOutput)
    
    try:
        res = structured_llm.invoke([
            SystemMessage(content="Rate the code completeness and compliance based on the QA report. Return a structured score 1 to 10."),
            HumanMessage(content=qa_report)
        ])
        score = res.score
    except Exception:
        print("[Scorer] LLM failed, using default score 5/10")
        score = 5  # Safe default if parsing fails
        
    print(f"\n[Scorer Node] Assigned Score: {score}/10")
    
    scores = state.get("iteration_score", [])
    return {
        "iteration_score": scores + [score], 
        "current_actor": "scorer"
    }

## Task 3: Advanced Memory & Context Management
We implement a context summarizer middleware to compress message history once a token threshold is breached or when we hit the mandatory minimum 3 iterations.

In [0]:
def summarizer_middleware(state: GameState):
    current_iter = state.get("iteration", 0)
    architect_msgs = state.get("architect_messages", [])
    qa_msgs = state.get("qa_feedback", [])
    
    # Estimate total token count (approx. 4 chars per token)
    total_chars = sum(len(m.content) for m in architect_msgs) + sum(len(m.content) for m in qa_msgs)
    token_est = total_chars // 4
    
    # Predefined threshold (e.g., 2000 tokens)
    threshold = 2000
    
    # We must also force summarization on/after iteration 2 to satisfy Task 3.1 requirement (3 iterations min, demonstrate summarization)
    if token_est > threshold or current_iter >= 2:
        print(f"\n[Summarizer Middleware] Context tokens ({token_est}) exceeded threshold ({threshold}) or iteration constraint reached. Condensing history...")
        
        # Summarize Architect Designs
        if architect_msgs:
            history_str = "\n\n".join([f"Design {i+1}:\n{m.content}" for i, m in enumerate(architect_msgs)])
            arch_summary_prompt = f"Summarize this software design history. Consolidate into a single concise final list of game system designs:\n\n{history_str}"
            arch_res = llm.invoke([SystemMessage(content="You are a context compression assistant."), HumanMessage(content=arch_summary_prompt)])
            new_architect = [AIMessage(content=f"[CONSOLIDATED ARCHITECT DESIGN SUMMARY]:\n{arch_res.content}")]
        else:
            new_architect = architect_msgs
            
        # Summarize QA Feedback
        if qa_msgs:
            feedback_str = "\n\n".join([f"Feedback {i+1}:\n{m.content}" for i, m in enumerate(qa_msgs)])
            qa_summary_prompt = f"Summarize this QA analysis history. Retain only outstanding bugs, validation failures, and design deviations:\n\n{feedback_str}"
            qa_res = llm.invoke([SystemMessage(content="You are a context compression assistant."), HumanMessage(content=qa_summary_prompt)])
            new_qa = [HumanMessage(content=f"[CONSOLIDATED QA FEEDBACK SUMMARY]:\n{qa_res.content}")]
        else:
            new_qa = qa_msgs
            
        print("[Summarizer Middleware] Context successfully summarized and replaced.")
        return {
            "architect_messages": new_architect,
            "qa_feedback": new_qa,
            "current_actor": "summarizer"
        }
        
    return {"current_actor": "summarizer"}

In [0]:
def director_review_node(state: GameState):
    print("\n================== DIRECTOR HITL REVIEW ==================")
    latest_score = state.get("iteration_score", [])[-1]
    current_iter = state.get("iteration", 0)
    print(f"Iteration: {current_iter} | Current Score: {latest_score}/10")
    
    # Task 3.1 requires executing at least 3 iterations
    if current_iter < 3:
        print(f"[HITL] Iteration {current_iter}/3. Forcing another iteration to demonstrate context summarization.")
        satisfied_choice = 'n'
        feedback = "Forcing iteration to meet the 3-iteration demonstration requirement."
    else:
        satisfied_choice = input("Are you satisfied with the generated game code? (y/n): ").strip().lower()
        if satisfied_choice == 'y':
            feedback = ""
        else:
            feedback = input("Please enter your custom feedback/revisions for the Architect: ").strip()
            
    if satisfied_choice == 'y':
        return {"satisfied": True, "current_actor": "director_feedback"}
    else:
        history = state.get("director_messages", [])
        # Append the new director feedback to director_messages history
        new_feedback_msg = HumanMessage(content=f"[Director Feedback - Iteration {current_iter}]: {feedback}")
        return {
            "satisfied": False, 
            "director_messages": history + [new_feedback_msg],
            "current_actor": "director_feedback"
        }

## Task 3: StateGraph & Checkpoint Compiler
We register the nodes and setup conditional routing. The graph is compiled with persistent checkpointers and interrupts before every individual agent (`architect`, `engineer`, `qa`, `director_review`).

In [0]:
def routing_edge(state: GameState):
    if state.get("satisfied", False):
        return END
    return "architect"

workflow = StateGraph(GameState)

# Register Nodes
workflow.add_node("director", director_node)
workflow.add_node("architect", architect_node)
workflow.add_node("engineer", engineer_node)
workflow.add_node("qa", qa_node)
workflow.add_node("scorer", score_node)
workflow.add_node("summarizer", summarizer_middleware)
workflow.add_node("director_feedback", director_review_node)

# Add Edges
workflow.add_edge(START, "director")
workflow.add_edge("director", "architect")
workflow.add_edge("architect", "engineer")
workflow.add_edge("engineer", "qa")
workflow.add_edge("qa", "scorer")
workflow.add_edge("scorer", "summarizer")
workflow.add_edge("summarizer", "director_feedback")

# Routing after HITL node
workflow.add_conditional_edges("director_feedback", routing_edge, {END: END, "architect": "architect"})

# Configure memory checkpointing and agent interrupt boundaries
memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["architect", "engineer", "qa", "director_feedback"]
)

## Task 4: System Execution & MLflow Tracking
We execute the compiled graph. When interrupted, the user is notified of the boundary and can view progress. After the HITL node executes, a child run is logged to MLflow capturing the iteration details, latencies, groundedness metrics, and game script.

In [0]:
def calculate_groundedness(plan: str, code: str) -> float:
    """
    LLM-as-a-judge metric to evaluate how accurately the generated code implements the design requirements.
    Returns a score between 0.0 and 1.0.
    """
    prompt = (
        "Evaluate the groundedness (accuracy of implementation) of the Python Pygame code against the Architect's Plan.\n"
        "Check if OOP design, Day/Night changes, Clustered obstacles, Speed increases, and High score saving are present.\n"
        "Plan:\n"
        f"{plan}\n\n"
        "Code:\n"
        f"{code}\n\n"
        "Rate groundedness on a scale of 0.0 (completely ungrounded) to 1.0 (perfectly grounded/compliant).\n"
        "Your response must be ONLY a single floating-point number, e.g., 0.95"
    )
    try:
        response = llm.invoke([SystemMessage(content="You are a strict code evaluator."), HumanMessage(content=prompt)])
        score_str = response.content.strip()
        # Handle empty or malformed responses
        if not score_str:
            print("[Warning] Empty response from LLM in calculate_groundedness. Using default 0.5")
            return 0.5
        # Try to extract float from response
        score = float(score_str)
        # Clamp to valid range
        return max(0.0, min(1.0, score))
    except (ValueError, AttributeError) as e:
        print(f"[Warning] Failed to parse groundedness score: {e}. Using default 0.5")
        return 0.5
    except Exception as e:
        print(f"[Warning] Unexpected error in calculate_groundedness: {e}. Using default 0.5")
        return 0.5

# Set thread session config for checkpointer
config = {"configurable": {"thread_id": "dino_runner_pa5_session"}}

# Define MLflow Experiment
mlflow.set_experiment("/Users/25280019@lums.edu.pk/PA5_Experiment")

# Run execution
print("--- Starting PA5 LangGraph Workflow ---")
initial_state = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": "",
    "qa_feedback": [],
    "current_actor": "",
    "iteration": 0,
    "iteration_score": [],
    "latencies": {},
    "satisfied": False
}

try:
    with mlflow.start_run(run_name="PA5_Pipeline_Run") as parent_run:
        # Initialize or resume graph stream
        state_snap = app.get_state(config)
        if not state_snap.values:
            # Start graph stream
            for event in app.stream(initial_state, config=config):
                pass
        
        retry_count = 0
        max_retries = 3
        
        while True:
            state_snap = app.get_state(config)
            
            # Break if the graph execution is completed
            if not state_snap.next:
                break
                
            next_agent = state_snap.next[0]
            print(f"\n[HITL Pause] Halting execution before node: '{next_agent}'")
            
            # Display context updates before resuming
            if next_agent == "architect" and state_snap.values.get("director_messages"):
                print(f"Latest Director Prompt: '{state_snap.values['director_messages'][-1].content}'")
            elif next_agent == "engineer" and state_snap.values.get("architect_messages"):
                print(f"Latest Architect Design (truncated): '{state_snap.values['architect_messages'][-1].content[:200]}...'")
            elif next_agent == "qa" and state_snap.values.get("engineer_code"):
                print("Code generated successfully. Ready to run QA.")
            elif next_agent == "director_feedback" and state_snap.values.get("iteration_score"):
                print(f"Scorer assigned score: {state_snap.values['iteration_score'][-1]}/10")
                
            input("Press Enter to resume execution and proceed...")
            
            # Resume the graph with error handling
            try:
                for event in app.stream(None, config=config):
                    pass
                retry_count = 0  # Reset retry counter on success
            except Exception as stream_error:
                retry_count += 1
                print(f"\n[Warning] Stream event error (attempt {retry_count}/{max_retries}): {type(stream_error).__name__}: {stream_error}")
                
                if retry_count >= max_retries:
                    print(f"\n[Error] Max retries ({max_retries}) reached. The workflow cannot proceed.")
                    print("Possible issues:")
                    print("  - LLM endpoint is returning malformed responses")
                    print("  - The prompt may be too complex or too long")
                    print("  - There may be a temporary API issue")
                    print("\nRecommendations:")
                    print("  1. Wait a few minutes and run this cell again to resume from checkpoint")
                    print("  2. Check if the LLM endpoint is available")
                    print("  3. Consider simplifying the prompts in the agent nodes")
                    break
                    
                print("Retrying from checkpoint...")
                continue
                
            # Logging iteration results after director_feedback runs
            updated_state = app.get_state(config).values
            if updated_state.get("current_actor") == "director_feedback":
                current_iter = updated_state.get("iteration", 0)
                score = updated_state.get("iteration_score", [])[-1]
                code = updated_state.get("engineer_code", "")
                plan = updated_state.get("architect_messages", [])[-1].content if updated_state.get("architect_messages") else ""
                
                # Compute groundedness score
                groundedness = calculate_groundedness(plan, code)
                
                # Start Nested MLflow Run for this iteration
                try:
                    with mlflow.start_run(run_name=f"Iteration_{current_iter}", nested=True):
                        mlflow.log_metric("groundedness", groundedness)
                        mlflow.log_metric("score", score)
                        
                        # Fetch latencies
                        latencies = updated_state.get("latencies", {})
                        arch_latency = latencies.get(f"architect_iter_{current_iter}", 0)
                        eng_latency = latencies.get(f"engineer_iter_{current_iter}", 0)
                        qa_latency = latencies.get(f"qa_iter_{current_iter}", 0)
                        
                        mlflow.log_metric("latency_architect", arch_latency)
                        mlflow.log_metric("latency_engineer", eng_latency)
                        mlflow.log_metric("latency_qa", qa_latency)
                        mlflow.log_metric("iteration_latency", arch_latency + eng_latency + qa_latency)
                        
                        # Log code as artifact
                        mlflow.log_text(code, f"dino_runner_iter_{current_iter}.py")
                        
                    print(f"\n[MLflow] Logged Iteration {current_iter} run. Groundedness={groundedness}, Latency={arch_latency+eng_latency+qa_latency:.2f}s")
                except Exception as e:
                    print(f"\n[Warning] Failed to log iteration {current_iter} to MLflow: {e}")

        # Extract final code and write out
        final_state = app.get_state(config).values
        final_code = final_state.get("engineer_code", "")
        if final_code:
            with open("25280019_dino_runner.py", "w", encoding="utf-8") as f:
                f.write(final_code)
            print("\n--- Pipeline Completed Successfully! ---")
            print("Final game script written to '25280019_dino_runner.py'")
        else:
            print("\n--- Pipeline Ended ---")
            print("No final code was generated.")
        
except KeyboardInterrupt:
    print("\n[Interrupted] Execution cancelled by user.")
    print("Progress has been checkpointed. You can resume by running this cell again.")
except Exception as e:
    print(f"\n[Error] Pipeline execution failed: {type(e).__name__}: {e}")
    print("Progress has been checkpointed. Check the error above and run this cell again to resume.")
    raise

# Director Prompt: Create a fully functional, highly polished endless runner game in Python using Pygame, inspired by the Chrome Dino game. The game MUST be built using strict Object-Oriented Programming (OOP) principles with dedicated classes for the Dino, Obstacles, and the Score. Include flying obstacles (Pterodactyls) and ground obstacles (Cacti) that spawn dynamically using pygame.time.set_timer(). Do NOT spawn a cactus and pterodactyl at the exact same time; randomize their spawn independently. Cacti MUST spawn at the exact bottom of the screen (y = SCREEN_HEIGHT - height), while Pterodactyls fly at random heights. Use distinct RGB colors for different entities (e.g., Green cacti, Red pterodactyls, Grey dino). The Dino must feature accurate jump/fall physics and ducking mechanics bound to pygame.KEYDOWN that seamlessly restore the original height upon pygame.KEYUP. On collision, display a 'Game Over' screen and wait for SPACE to trigger a perfect reset. Crucially: Do NOT use a timer for the score. The score MUST increment by 1 ONLY when an obstacle successfully passes the Dino. The Score class MUST also track a persistent 'High Score' that is updated on collision and displayed on the Game Over screen alongside the current score.

--- Starting PA5 LangGraph Workflow ---


Awaiting Director Prompt:  Create a fully functional, highly polished endless runner game in Python using Pygame, inspired by the Chrome Dino game. The game MUST be built using strict Object-Oriented Programming (OOP) principles with dedicated classes for the Dino, Obstacles, and the Score. Include flying obstacles (Pterodactyls) and ground obstacles (Cacti) that spawn dynamically using pygame.time.set_timer(). Do NOT spawn a cactus and pterodactyl at the exact same time; randomize their spawn independently. Cacti MUST spawn at the exact bottom of the screen (y = SCREEN_HEIGHT - height), while Pterodactyls fly at random heights. Use distinct RGB colors for different entities (e.g., Green cacti, Red pterodactyls, Grey dino). The Dino must feature accurate jump/fall physics and ducking mechanics bound to pygame.KEYDOWN that seamlessly restore the original height upon pygame.KEYUP. On collision, display a 'Game Over' screen and wait for SPACE to trigger a perfect reset. Crucially: Do NOT 


[HITL Pause] Halting execution before node: 'architect'
Latest Director Prompt: 'Create a fully functional, highly polished endless runner game in Python using Pygame, inspired by the Chrome Dino game. The game MUST be built using strict Object-Oriented Programming (OOP) principles with dedicated classes for the Dino, Obstacles, and the Score. Include flying obstacles (Pterodactyls) and ground obstacles (Cacti) that spawn dynamically using pygame.time.set_timer(). Do NOT spawn a cactus and pterodactyl at the exact same time; randomize their spawn independently. Cacti MUST spawn at the exact bottom of the screen (y = SCREEN_HEIGHT - height), while Pterodactyls fly at random heights. Use distinct RGB colors for different entities (e.g., Green cacti, Red pterodactyls, Grey dino). The Dino must feature accurate jump/fall physics and ducking mechanics bound to pygame.KEYDOWN that seamlessly restore the original height upon pygame.KEYUP. On collision, display a 'Game Over' screen and wait f

Press Enter to resume execution and proceed... 


========== ITERATION 1 ==========

[HITL Pause] Halting execution before node: 'engineer'
Latest Architect Design (truncated): '**Dino Runner Game System Architecture**

### Class Diagram

The game will be designed using the following classes:

*   `Dino`: Represents the dinosaur characte...'


...


[QA Node] Report generated successfully (1677 chars)

[Scorer Node] Assigned Score: 6/10

[Summarizer Middleware] Context tokens (3210) exceeded threshold (2000) or iteration constraint reached. Condensing history...
[Summarizer Middleware] Context successfully summarized and replaced.

[HITL Pause] Halting execution before node: 'director_feedback'
Scorer assigned score: 6/10


Press Enter to resume execution and proceed... 


================== DIRECTOR HITL REVIEW ==================
Iteration: 3 | Current Score: 6/10


Are you satisfied with the generated game code? (y/n):  y


[MLflow] Logged Iteration 3 run. Groundedness=1.0, Latency=44.10s

--- Pipeline Completed Successfully! ---
Final game script written to '25280019_dino_runner.py'
